In [1]:
!pip install transformers datasets torch evaluate peft accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# =====================================================================================
# Ensure all dependencies are installed in your environment first.
# In a Kaggle/Colab notebook, you would run:
# !pip install transformers datasets torch evaluate peft accelerate -q
# =====================================================================================

import torch
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# =====================================================================================
# 1. Configuration and Initialization
# =====================================================================================

MODEL_CHECKPOINT = "distilbert-base-uncased"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# =====================================================================================
# 2. Dataset Loading and Preprocessing
# =====================================================================================

imdb_dataset = load_dataset("imdb")
train_dataset = imdb_dataset["train"].shuffle(seed=42)
test_dataset = imdb_dataset["test"].shuffle(seed=42)

print(f"\nTrain dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print("\nDataset sample:")
print(train_dataset[0])

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True, max_length=512)

tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
tokenized_test_dataset = test_dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =====================================================================================
# 3. Evaluation Metrics
# =====================================================================================

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {"accuracy": accuracy["accuracy"], "f1": f1["f1"]}

# =====================================================================================
# Experiment 0: Pretrained Model Baseline (No Fine-Tuning)
# =====================================================================================
print("\n" + "="*50)
print("Experiment 0: Pretrained Model Baseline Evaluation")
print("="*50)

no_finetuning_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2
).to(DEVICE)

baseline_args = TrainingArguments(
    output_dir="./results/baseline_eval",
    per_device_eval_batch_size=16,
    report_to="none"
)

baseline_trainer = Trainer(
    model=no_finetuning_model,
    args=baseline_args,
    eval_dataset=tokenized_test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\nEvaluating pretrained model without fine-tuning...")
no_finetuning_eval_results = baseline_trainer.evaluate()
print(f"Pretrained Model Evaluation Results: {no_finetuning_eval_results}")

# =====================================================================================
# Experiment 1: Full Fine-Tuning
# =====================================================================================
print("\n" + "="*50)
print("Experiment 1: Starting Full Fine-Tuning")
print("="*50)

full_finetuning_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2
).to(DEVICE)

# Define training hyperparameters for full fine-tuning
full_finetuning_args = TrainingArguments(
    output_dir="./results/full_finetuning",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,                   # <-- MODIFIED: Changed from 1 to 3
    weight_decay=0.01,
    logging_strategy="epoch",             # <-- ADDED: Log training loss at the end of each epoch
    eval_strategy="epoch",                # Evaluate after every training epoch
    save_strategy="epoch",                # Save model after every epoch
    load_best_model_at_end=True,          # Keep the best model (based on eval metrics)
    report_to="none"
)

full_finetuning_trainer = Trainer(
    model=full_finetuning_model,
    args=full_finetuning_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\nStarting full fine-tuning on the complete dataset for 3 epochs...")
full_finetuning_trainer.train()

print("\nEvaluating final best full fine-tuning model...")
full_finetuning_eval_results = full_finetuning_trainer.evaluate()
print(f"Full Fine-Tuning Evaluation Results: {full_finetuning_eval_results}")

# =====================================================================================
# Experiment 2: Parameter-Efficient Fine-Tuning (LoRA)
# =====================================================================================
print("\n" + "="*50)
print("Experiment 2: Starting Fine-Tuning with LoRA")
print("="*50)

base_model_for_lora = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=4,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "k_lin", "v_lin"]
)

lora_model = get_peft_model(base_model_for_lora, lora_config).to(DEVICE)
print("\nLoRA Model Parameter Analysis:")
lora_model.print_trainable_parameters()

lora_training_args = TrainingArguments(
    output_dir="./results/lora_finetuning",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,                   # <-- MODIFIED: Changed from 1 to 3
    weight_decay=0.01,
    logging_strategy="epoch",             # <-- ADDED: Log training loss at the end of each epoch
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\nStarting LoRA fine-tuning on the complete dataset for 3 epochs...")
lora_trainer.train()

print("\nEvaluating final best LoRA fine-tuning model...")
lora_eval_results = lora_trainer.evaluate()
print(f"LoRA Fine-Tuning Evaluation Results: {lora_eval_results}")

# =====================================================================================
# Summary of Experiment Results
# =====================================================================================
print("\n" + "="*50)
print("Summary of Experiment Results")
print("="*50)

print(f"Pretrained (no fine-tuning) -> Accuracy: {no_finetuning_eval_results['eval_accuracy']:.4f}, F1 Score: {no_finetuning_eval_results['eval_f1']:.4f}")
print(f"Full fine-tuning (best model after 3 epochs) -> Accuracy: {full_finetuning_eval_results['eval_accuracy']:.4f}, F1 Score: {full_finetuning_eval_results['eval_f1']:.4f}")
print(f"LoRA fine-tuning (best model after 3 epochs) -> Accuracy: {lora_eval_results['eval_accuracy']:.4f}, F1 Score: {lora_eval_results['eval_f1']:.4f}")

2025-10-15 17:03:00.592433: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760547780.775989      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760547780.825561      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_19/1504095405.py:195: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  lora_trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.



LoRA Model Parameter Analysis:
trainable params: 702,722 || all params: 67,657,732 || trainable%: 1.0386

Starting LoRA fine-tuning on the complete dataset for 3 epochs...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.335700,0.297294,0.894040,0.893985
2,0.281000,0.262422,0.905400,0.905394
3,0.270200,0.267575,0.906880,0.906875



Evaluating final best LoRA fine-tuning model...


LoRA Fine-Tuning Evaluation Results: {'eval_loss': 0.2624223828315735, 'eval_accuracy': 0.9054, 'eval_f1': 0.9053935139306853, 'eval_runtime': 210.874, 'eval_samples_per_second': 118.554, 'eval_steps_per_second': 14.819, 'epoch': 3.0}

Summary of Experiment Results
Pretrained (no fine-tuning) -> Accuracy: 0.5305, F1 Score: 0.5077
Full fine-tuning (best model after 3 epochs) -> Accuracy: 0.9279, F1 Score: 0.9279
LoRA fine-tuning (best model after 3 epochs) -> Accuracy: 0.9054, F1 Score: 0.9054


In [3]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
print("\n" + "="*50)
print("Starting Interactive Sentiment Prediction")
print("="*50)

print("Loading the trained LoRA model for inference...")
best_lora_model_path = lora_trainer.state.best_model_checkpoint
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)
inference_model = PeftModel.from_pretrained(base_model, best_lora_model_path).to(DEVICE)
inference_model.eval()
print(f"Model loaded from: {best_lora_model_path}")

def predict_sentiment(text, model, tokenizer, device):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=-1).item()
    label_map = {0: "Negative Sentiment", 1: "Positive Sentiment"}
    return label_map[prediction]

test_sentences = [
    "This movie was a complete waste of my time. The acting was terrible and the plot was boring.",
    "I absolutely loved this film! The visuals were stunning and the story was heartwarming.",
    "The book is always better than the movie.",
    "It was  bad, actually quite boring in some parts.",
    "A masterpiece of modern cinema. I was captivated from beginning to end."
]

print("\nStarting test on sample sentences:")
for sentence in test_sentences:
    prediction = predict_sentiment(sentence, inference_model, tokenizer, DEVICE)
    print(f"  - Sentence: \"{sentence}\"")
    print(f"    Prediction: {prediction}\n")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting Interactive Sentiment Prediction
Loading the trained LoRA model for inference...
Model loaded from: ./results/lora_finetuning/checkpoint-6250

Starting test on sample sentences:
  - Sentence: "This movie was a complete waste of my time. The acting was terrible and the plot was boring."
    Prediction: Negative Sentiment

  - Sentence: "I absolutely loved this film! The visuals were stunning and the story was heartwarming."
    Prediction: Positive Sentiment

  - Sentence: "The book is always better than the movie."
    Prediction: Positive Sentiment

  - Sentence: "It was  bad, actually quite boring in some parts."
    Prediction: Negative Sentiment

  - Sentence: "A masterpiece of modern cinema. I was captivated from beginning to end."
    Prediction: Positive Sentiment

